In [ ]:
# Jupyter cell — draw two line scans with smooth curved connectors
#
# CSV columns expected:
#   1) x coordinate: Distance_(pixels)
#   2) y value:      Gray_Value
#   3) group label:  Region
#
# Behavior:
#   - Plots the two groups in the third column
#   - Uses round scatter points
#   - Colors points with custom LUT.csv if available
#   - Uses a 0.0037 to 0.6 color scale
#   - Connects points with a smooth shape-preserving curved line (PCHIP)
#   - Keeps the plot area square
#   - Exports an Adobe Illustrator-friendly PDF

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, ListedColormap
from scipy.interpolate import PchipInterpolator

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

# ============================================================
# Settings
# ============================================================
BASE_DIR = Path(globals().get("SCRIPT_DIR", Path.cwd()))
CSV_PATH = BASE_DIR / "linescan_S21Ar1andS23Ar1.csv"
OUT_DIR = BASE_DIR / "output_linescans"
OUT_DIR.mkdir(parents=True, exist_ok=True)

X_COL = "Distance_(pixels)"
Y_COL = "Gray_Value"
GROUP_COL = "Region"

GLOBAL_VMIN = 0.0037
GLOBAL_VMAX = 0.6

# ============================================================
# Load CSV
# ============================================================
if not CSV_PATH.exists():
    raise FileNotFoundError(f"Could not find input CSV: {CSV_PATH}")

df = pd.read_csv(CSV_PATH)

required_cols = [X_COL, Y_COL, GROUP_COL]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df[X_COL] = pd.to_numeric(df[X_COL], errors="coerce")
df[Y_COL] = pd.to_numeric(df[Y_COL], errors="coerce")
df[GROUP_COL] = df[GROUP_COL].astype(str)

df = df.dropna(subset=[X_COL, Y_COL, GROUP_COL]).copy()

groups = list(df[GROUP_COL].dropna().unique())
if len(groups) != 2:
    print(f"Warning: expected 2 groups, found {len(groups)}: {groups}")

# ============================================================
# Load custom LUT.csv if present
# ============================================================
def load_fiji_lut_try(paths=(BASE_DIR / "LUT.csv", CSV_PATH.parent / "LUT.csv")):
    for p in paths:
        try:
            if not p.exists():
                continue

            lut_df = pd.read_csv(p)
            cols = [c.strip().lower() for c in lut_df.columns]

            if {"red", "green", "blue"}.issubset(set(cols)):
                r_idx = cols.index("red")
                g_idx = cols.index("green")
                b_idx = cols.index("blue")
            elif {"r", "g", "b"}.issubset(set(cols)):
                r_idx = cols.index("r")
                g_idx = cols.index("g")
                b_idx = cols.index("b")
            else:
                r_idx, g_idx, b_idx = 0, 1, 2

            rgb = lut_df.iloc[:, [r_idx, g_idx, b_idx]].to_numpy(dtype=float) / 255.0
            rgb = np.clip(rgb, 0.0, 1.0)
            return ListedColormap(rgb, name="custom_lut")
        except Exception as e:
            print(f"Could not load LUT from {p}: {e}")

    return None

CUSTOM_CMAP = load_fiji_lut_try()
CMAP = CUSTOM_CMAP if CUSTOM_CMAP is not None else plt.get_cmap("viridis")
NORM = Normalize(vmin=GLOBAL_VMIN, vmax=GLOBAL_VMAX)

# ============================================================
# Plot
# ============================================================
fig, ax = plt.subplots(figsize=(7.5, 7.), constrained_layout=True)
ax.set_box_aspect(1)

for group in groups:
    gdf = df[df[GROUP_COL] == group].sort_values(X_COL).copy()

    x = gdf[X_COL].to_numpy(dtype=float)
    y = gdf[Y_COL].to_numpy(dtype=float)

    if len(x) == 0:
        continue

    # Round scatter points colored by y-value
    ax.scatter(
        x,
        y,
        s=70,
        marker="o",
        c=y,
        cmap=CMAP,
        norm=NORM,
        edgecolors="black",
        linewidths=0.25,
        alpha=0.95,
        label=group,
        zorder=3,
    )

    # Smooth curved connector: PCHIP avoids overshoot and looks good for scan traces
    if len(x) >= 3 and np.isfinite(x).all() and np.isfinite(y).all():
        try:
            curve = PchipInterpolator(x, y)
            x_fit = np.linspace(np.min(x), np.max(x), max(200, len(x) * 10))
            y_fit = curve(x_fit)

            ax.plot(
                x_fit,
                y_fit,
                linewidth=2.0,
                alpha=0.95,
                zorder=4,
                label=f"{group} smooth curve",
            )
        except Exception:
            # Fallback to simple point-to-point line if interpolation fails
            ax.plot(
                x,
                y,
                linewidth=2.0,
                alpha=0.95,
                zorder=4,
                label=f"{group} line",
            )
    elif len(x) >= 2:
        ax.plot(
            x,
            y,
            linewidth=2.0,
            alpha=0.95,
            zorder=4,
            label=f"{group} line",
        )

ax.set_ylim(0, 0.6)

# Colorbar
sm = mpl.cm.ScalarMappable(norm=NORM, cmap=CMAP)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, fraction=0.046, pad=0.03)
cbar.set_label(Y_COL)

# Labels / style
ax.set_xlabel(X_COL)
ax.set_ylabel(Y_COL)
ax.set_title("Line scans with smooth curved connectors")
ax.grid(alpha=0.18)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(frameon=False, fontsize=9)

# ============================================================
# Save as Illustrator-friendly PDF
# ============================================================
out_pdf = OUT_DIR / f"{CSV_PATH.stem}_linescan.pdf"
fig.savefig(
    out_pdf,
    format="pdf",
    bbox_inches="tight",
    facecolor="white",
    edgecolor="none",
)
plt.show(fig)
plt.close(fig)

print(f"Saved PDF to: {out_pdf}")